RETRIEVER

In [1]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader,get_response_synthesizer
from llama_index.core import SimpleDirectoryReader
from llama_index.core import Settings

Settings.chunk_size = 128
Settings.chunk_overlap = 50

documents = SimpleDirectoryReader("http://farm8.staticflickr.com/7123/7683270566_22bff11f7d_z.jpg").load_data()
index = VectorStoreIndex.from_documents(documents)
retriever = index.as_retriever(verbose=True, similarity_top_k=2)
response_synthesizer = get_response_synthesizer(
    response_mode="compact",
)
query_engine = index.as_query_engine()

ValueError: Directory http://farm8.staticflickr.com/7123/7683270566_22bff11f7d_z.jpg does not exist.

In [2]:
eval_q_data = SimpleDirectoryReader("/mnt/disk1/sjw/llama_index/eval-q").load_data()

In [3]:
q_data = eval_q_data[0].text

questions = [line.strip().strip('"') for line in q_data.split('\n') if line.strip()]

In [4]:
retriever_result = {}
search_results = []

for count, question in enumerate(questions, start=1):
    search_this = f"{question}"
    search_result = retriever.retrieve(search_this)
    search_results = []
    
    for i in range(retriever.similarity_top_k):
        search_results.append(search_result[i].get_text())

    retriever_result[count] = search_results

EMBEDDING MODEL

In [5]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

GENERATOR

In [6]:
from llama_index.core import VectorStoreIndex

vector_index = VectorStoreIndex.from_documents(documents)

In [7]:
responses=[]

In [8]:
for count, question in enumerate(questions, start=1):
    retriever_context_list = retriever_result.get(count, [])
    retriever_context = ' '.join(retriever_context_list)
    
    query = f"Given the context: '{retriever_context}', determine if the following statement is 'True' or 'False': {question}. Start your response with 'True' or 'False'. Answer:"

    # It's better to create the query engine once outside the loop if it doesn't need to be recreated each time
    query_engine = index.as_query_engine()
    response = query_engine.query(query)

    # Assuming response has a 'text' attribute that contains the response as a string
    response_text = response.text if hasattr(response, 'text') else str(response)

    # Find the position of "Answer:" and extract the response
    #aa = response_text.find("Answer:")
    #if aa != -1:
     #   response = response_text[aa + 7:].strip().split()[0]  # Extract the first word after "Answer:"

    responses.append(response)

    print(count, response)

1 True
2 False.
3 True
4 False.
5 False
6 True
7 False
8 True
9 False
10 False
11 False
12 True
13 False
14 True
15 False
16 True
17 False
18 True
19 False
20 True
21 False.
22 True
23 False
24 False.
25 True
26 False.
27 True
28 False.
29 True
30 True
31 False.
32 True
33 True
34 False.
35 True
36 True
37 True
38 False.
39 True
40 False
41 True
42 False.
43 False
44 True
45 True
46 False
47 True
48 False
49 True
50 False
51 True
52 False.
53 False
54 True
55 True
56 True
57 False.
58 True
59 False.
60 False
61 True
62 True
63 True
64 False
65 True
66 True
67 True
68 True
69 True
70 True
71 False
72 True
73 True
74 True
75 True
76 False
77 True
78 False
79 False
80 False
81 False
82 False.
83 False
84 False
85 True
86 False
87 False
88 False.
89 False.
90 False
91 False
92 False
93 False
94 True
95 True
96 False
97 True
98 False
99 False
100 False
101 False.
102 False
103 False
104 False
105 False
106 False
107 False
108 False
109 True
110 False
111 True
112 True
113 False
114 True
115

EVALUATOR

In [10]:
eval_a_data = SimpleDirectoryReader("/mnt/disk1/sjw/llama_index/eval-a").load_data()

In [11]:
a_data = eval_a_data[0].text

answers = [line.strip().strip('"') for line in a_data.split('\n') if line.strip()]

In [12]:
responses_str = []

In [13]:
for response in responses:
    response_str = str(response)
    if "True" in response_str:
        response_str = "True"
    elif "False" in response_str:
        response_str = "False"
    
    responses_str.append(response_str)

In [14]:
correct_count=0
number=0

for answer, response, response_str, question in zip(answers, responses, responses_str, questions,):
    number+=1

    if answer == response_str:
        correct_count += 1
    else: 
        print("<<wrong>>\n", number, question)
        print("RESPONSE", response)
        print("CORRECT ANSWER", answer)
        print()
print(f"correct_count: {correct_count}")

<<wrong>>
 5 True or False: Scientists plan to use particle colliders to manipulate the fourth dimension.
RESPONSE False
CORRECT ANSWER True

<<wrong>>
 7 True or False: Humanity's understanding of three-dimensional space limited the potential for technological innovations before the discovery of the fourth dimension.
RESPONSE False
CORRECT ANSWER True

<<wrong>>
 10 True or False: The discovery of exotic matter with negative mass is a direct result of fourth-dimensional research.
RESPONSE False
CORRECT ANSWER True

<<wrong>>
 23 True or False: Human genetic modifications are necessary for extracting oxygen from a liquid atmosphere due to fourth-dimensional advancements.
RESPONSE False
CORRECT ANSWER True

<<wrong>>
 37 True or False: The new breathable liquid atmosphere will increase the friction in oxygen delivery.
RESPONSE True
CORRECT ANSWER False

<<wrong>>
 64 True or False: Advanced nanotechnology is essential for maintaining the proper oxygen levels in Earth's liquid atmosphere

In [15]:
accuracy = (correct_count / len(questions)) * 100
print(f"Total Questions: {len(questions)}")
print(f"Correct Answers: {correct_count}")
print(f"Accuracy: {accuracy:.2f}%")

Total Questions: 200
Correct Answers: 160
Accuracy: 80.00%


In [ ]:
Settings